# 00renfegtfs: Renfe's high-speed and long-distance timetable (GTFS)

**What it does:** downloads Renfe's official GTFS feed (AVE, Avlo, Alvia, Avant, Intercity and Media Distancia trains), unpacks it, and takes a first look: the files and their row counts, the dates the timetable covers, and the stations whose names match our 32 cities. Two versions are used: today's feed from Renfe, and an archived copy from 18 December 2025 kept by the Mobility Database (the timetable as it stood before the speed restrictions of 2026).

**Input:** the internet (Renfe's server and files.mobilitydatabase.org).

**Output:** `data/raw/renfe/<download date>/` for each version, with the zip and its unpacked text files. Nothing is cleaned yet; later notebooks build the tables from these files.

**Data:** Renfe Viajeros, "Horarios de alta velocidad, larga distancia y media distancia", Renfe open data portal (data.renfe.com), under Renfe's open data terms. Archived copy: Mobility Database (MobilityData), feed mdb-2620.

## 0. Settings

In [1]:
from pathlib import Path
import datetime
import unicodedata
import zipfile
import requests
import pandas as pd

RAW_DIR = Path("data") / "raw" / "renfe"  # <- downloads go here, one subfolder per version
FEEDS = {}
FEEDS["2026-11-16"] = "https://ssl.renfe.com/gtransit/Fichero_AV_LD/google_transit.zip"  # <- today's feed from Renfe (folder name = day it was downloaded)
FEEDS["2025-12-18"] = "https://files.mobilitydatabase.org/mdb-2620/mdb-2620-202512180143/mdb-2620-202512180143.zip"  # <- archived feed of 18 Dec 2025 (Mobility Database)
STATION_CHECK_FEED = "2026-11-16"  # <- section 4 lists stations for this version only
CITY_WORDS = ["madrid", "sevilla", "cordoba", "ciudad real", "cadiz", "malaga", "granada", "zaragoza", "lleida", "tarragona", "barcelona", "girona", "valencia", "albacete", "alicante", "alacant", "murcia", "valladolid", "leon", "oviedo", "ourense", "santiago", "coruna", "vigo", "badajoz", "teruel", "almeria", "jaen", "huelva", "santander", "bilbao", "donostia", "san sebastian", "pamplona"]  # <- words searched in station names (lower case, no accents)

## 1. Download (skipped for versions already there)

In [2]:
for version, url in FEEDS.items():
    folder = RAW_DIR / version
    folder.mkdir(parents=True, exist_ok=True)
    zip_path = folder / "google_transit.zip"
    if zip_path.exists():
        print(version, "already downloaded:", zip_path)
        continue
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=180)
    response.raise_for_status()
    zip_path.write_bytes(response.content)
    print(version, "downloaded", round(len(response.content) / 1e6, 1), "MB to", zip_path)

2026-11-16 already downloaded: data\raw\renfe\2026-11-16\google_transit.zip
2025-12-18 downloaded 1.1 MB to data\raw\renfe\2025-12-18\google_transit.zip


## 2. Unpack and list the files of each version

In [3]:
def read_gtfs(folder, name):
    table = pd.read_csv(Path(folder) / name, dtype=str, keep_default_na=False, encoding="utf-8-sig", skipinitialspace=True)
    table.columns = [c.strip() for c in table.columns]
    for column in table.columns:
        table[column] = table[column].str.strip()
    return table

names_by_version = {}
for version in FEEDS:
    folder = RAW_DIR / version
    with zipfile.ZipFile(folder / "google_transit.zip") as z:
        z.extractall(folder)
        names_by_version[version] = z.namelist()
    print()
    print("Version", version)
    for name in names_by_version[version]:
        table = read_gtfs(folder, name)
        print(f"  {name:22s} {len(table):8d} rows   columns: {', '.join(table.columns)}")


Version 2026-11-16
  agency.txt                    1 rows   columns: agency_id, agency_name, agency_url, agency_timezone, agency_lang, agency_phone
  calendar_dates.txt        24619 rows   columns: service_id, date, exception_type
  calendar.txt               6402 rows   columns: service_id, monday, tuesday, wednesday, thursday, friday, saturday, sunday, start_date, end_date
  routes.txt                  671 rows   columns: route_id, agency_id, route_short_name, route_long_name, route_desc, route_type, route_url, route_color, route_text_color
  stops.txt                  1006 rows   columns: stop_id, stop_code, stop_name, stop_desc, stop_lat, stop_lon, zone_id, stop_url, location_type, parent_station, stop_timezone, wheelchair_boarding
  stop_times.txt            51846 rows   columns: trip_id, arrival_time, departure_time, stop_id, stop_sequence, stop_headsign, pickup_type, drop_off_type, shape_dist_traveled
  trips.txt                  6402 rows   columns: route_id, service_id, trip_

## 3. Dates covered by each version

In [4]:
for version, names in names_by_version.items():
    folder = RAW_DIR / version
    print()
    print("Version", version)
    if "calendar.txt" in names:
        calendar = read_gtfs(folder, "calendar.txt")
        print("  calendar.txt: services from", calendar["start_date"].min(), "to", calendar["end_date"].max())
    if "calendar_dates.txt" in names:
        calendar_dates = read_gtfs(folder, "calendar_dates.txt")
        print("  calendar_dates.txt: dates from", calendar_dates["date"].min(), "to", calendar_dates["date"].max())
        print("  exception types:", calendar_dates["exception_type"].value_counts().to_dict())
    if "agency.txt" in names:
        print("  operators:", list(read_gtfs(folder, "agency.txt")["agency_name"]))


Version 2026-11-16
  calendar.txt: services from 20261115 to 20270929
  calendar_dates.txt: dates from 20261116 to 20270124
  exception types: {'2': 24619}
  operators: ['RENFE OPERADORA']

Version 2025-12-18
  calendar.txt: services from 20251217 to 20261209
  calendar_dates.txt: dates from 20251218 to 20261208
  exception types: {'2': 68620}
  operators: ['RENFE OPERADORA']


## 4. Stations whose names match our cities (one version)

In [5]:
def plain(text):
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.lower()

stops = read_gtfs(RAW_DIR / STATION_CHECK_FEED, "stops.txt")
stops["plain_name"] = stops["stop_name"].map(plain)
print(len(stops), "stops in total")
for word in CITY_WORDS:
    found = stops[stops["plain_name"].str.contains(word, regex=False)]
    print()
    print(f"{word}: {len(found)} match(es)")
    if len(found) > 0:
        print(found[["stop_id", "stop_name", "stop_lat", "stop_lon"]].to_string(index=False))

1006 stops in total

madrid: 8 match(es)
stop_id                                stop_name   stop_lat   stop_lon
  17000         Madrid-Chamartín-Clara Campoamor 40.4711789 -3.6829524
  18000                  Madrid-Atocha Cercanías 40.4066191 -3.6894391
  60000 Madrid-Puerta de Atocha-Almudena Grandes 40.4064420 -3.6908860
  18002                Madrid-Nuevos Ministerios 40.4466271 -3.6923468
  18001                         Madrid-Recoletos 40.4233731 -3.6909310
  10000                      Madrid-Príncipe Pío 40.4214895 -3.7189689
  97201                     Madrid-Ramón y Cajal 40.4881473 -3.6948037
  70002             Asamblea de Madrid-Entrevías 40.3818618 -3.6680454

sevilla: 1 match(es)
stop_id           stop_name   stop_lat   stop_lon
  51003 Sevilla-Santa Justa 37.3924955 -5.9749269

cordoba: 4 match(es)
stop_id              stop_name   stop_lat   stop_lon
  50500  Córdoba-Julio Anguita 37.8882910 -4.7894530
  37704  Villanueva de Córdoba 38.2554049 -4.5709520
  50413     Alcol

___
**END**